In [1]:
!pip install -q gdown
!pip install pandas matplotlib seaborn tabulate
!gdown --folder 'https://drive.google.com/drive/folders/1aOIj8beykz7nNQkXATPwaFeRUVrrfjvB?usp=sharing'

Retrieving folder contents
Processing file 1fYmhcfNkN8oVqOSpVeKDrCJU-7F4B5qq adult_random_forest_entries_05-23_31-05-2025.txt
Processing file 1WAFNtthnb1tTsErEYApQGS2C-0vUuFHp adult_random_forest_info_05-23_31-05-2025.txt
Processing file 1tgtfqmUko-I5Zo2cmziL-xcEpkHY3WuR adult_random_forest_rules_05-23_31-05-2025.txt
Processing file 1R9hTGakgF0hPlIMJHlymhQ6ZcM9BeL6W adult_simpleNN_entries_30-23_31-05-2025.txt
Processing file 19imx8TfbcADCgY9F6avoQYtgYEgrbvX_ adult_simpleNN_info_30-23_31-05-2025.txt
Processing file 1X5NVmBiUQlA_zZidsZZJURN_TwH9RFmn adult_simpleNN_rules_30-23_31-05-2025.txt
Processing file 1Eos35YPFigetJxHT5fQcUG9-tKim56eh german_random_forest_entries_55-23_31-05-2025.txt
Processing file 1wJjs1UjXcswZzKzuHDrGjUU1Lnpw3aCs german_random_forest_info_55-23_31-05-2025.txt
Processing file 1I0i9Xqzfv49K2dqj5BacJVRd6yV3_GgO german_random_forest_rules_55-23_31-05-2025.txt
Processing file 1ZALJdVFXPCpLE3o1ZTxExHvIvVPIKwMs german_simpleNN_entries_14-00_01-06-2025.txt
Processing fil

In [2]:
import json
from collections import defaultdict
from tabulate import tabulate
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def load_entries_to_df(file_path):
    entries = []
    with open(file_path, "r") as f:
        for line in f:
            data = json.loads(line)
            if data.get("type") == "entry":
                instance_name = data.get("entry_name")
                instance = data.get("instance", {})
                features = instance.get("features", {})
                original = instance.get("original_outcome")
                predicted = instance.get("predicted_outcome")

                entry_dict = {
                    "Instance_Name": instance_name,
                    "Original_Outcome": original,
                    "Predicted_Outcome": predicted,
                    **features
                }
                entries.append(entry_dict)
    df = pd.DataFrame(entries)
    return df


def load_and_group_rules(file_path):
    grouped_rules = defaultdict(lambda: defaultdict(list))

    with open(file_path, "r") as f:
        for line in f:
            data = json.loads(line)
            if data.get("type") == "rule":
                premises = data["rule"]["premises"]
                premises_str = " AND ".join(f'{p["attr"]} {p["op"]} {p["val"]}' for p in premises)
                consequence = data["rule"]["consequence"]
                consequence_str = f'{consequence["attr"]} {consequence["op"]} {consequence["val"]}'
                rule_str = f'IF {premises_str} THEN {consequence_str}'

                metrics = data.get("metrics", {})
                rule_dict = {
                    "Premises": premises,
                    "Rule_ID": None,  # You can fill this later if needed
                    "Rule": rule_str,
                    "Cov": round(metrics.get("coverage", 0), 5),
                    "Cov_class": round(metrics.get("local_class_coverage", 0), 5),
                    "Pre": round(metrics.get("precision", 0), 5),
                    "Len": metrics.get("rule_length", 0),
                    "Reject": metrics.get("rejected", 0),
                    "Elapsed_time": round(metrics.get("elapsed_time", 0), 5),
                    "Iter_Limit": data.get("at_limit", False)
                }

                instance_name = data.get("entry_name")
                explainer = data.get("explainer")
                grouped_rules[instance_name][explainer].append(rule_dict)

    return grouped_rules

def grouped_rules_to_df(grouped_rules):
    rows = []
    for instance_name, explainers_dict in grouped_rules.items():
        for explainer, rules_list in explainers_dict.items():
           for i, rule_dict in enumerate(rules_list, start=1):
                row = {
                    "Instance_Name": instance_name,
                    "Explainer": explainer,
                }
                row.update(rule_dict)
                row["Rule_ID"] = f"{explainer}{i}"
                rows.append(row)
    return pd.DataFrame(rows)


def filter_non_dominated(df, cov_col='Cov', pre_col='Pre'):
    non_dominated_indices = []
    for i, row_i in df.iterrows():
        dominated = False
        for j, row_j in df.iterrows():
            if i != j:
                if (row_j[cov_col] >= row_i[cov_col] and
                    row_j[pre_col] >= row_i[pre_col] and
                    (row_j[cov_col] > row_i[cov_col] or row_j[pre_col] > row_i[pre_col])):
                    dominated = True
                    break
        if not dominated:
            non_dominated_indices.append(i)

    return df.loc[non_dominated_indices].reset_index(drop=True)


def filter_non_dominated_3d(df, cov_col='Cov_class', pre_col='Pre', len_col='Len'):
    # convert to numpy for speed
    data = df[[cov_col, pre_col, len_col]].to_numpy()

    non_dominated_mask = []
    for i, candidate in enumerate(data):
        dominated = False
        for j, other in enumerate(data):
            if i == j:
                continue
            # check if 'other' dominates 'candidate':
            # other is better or equal in all objectives AND strictly better in at least one
            # for cov_col and pre_col (to maximize): other >= candidate
            # for len_col (to minimize): other <= candidate
            if (other[0] >= candidate[0] and
                other[1] >= candidate[1] and
                other[2] <= candidate[2] and
                (other[0] > candidate[0] or other[1] > candidate[1] or other[2] < candidate[2])):
                dominated = True
                break
        non_dominated_mask.append(not dominated)

    return df.loc[non_dominated_mask]

def plot_non_dominated_rules(non_dominated_rules, instance_name):
    plt.figure(figsize=(8,6))
    sns.scatterplot(
        data=non_dominated_rules,
        x='Cov',
        y='Pre',
        hue='Explainer',
        style='Explainer',
        palette='tab10',
        s=100
    )

    plt.title(f"Non-dominated Rules for Instance {instance_name} (Coverage ↑, Precision ↑)")
    plt.xlabel("Coverage")
    plt.ylabel("Precision")
    plt.legend(title='Explainer', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def count_unique_attributes_from_rules(df_rules):
    attr_counter = Counter()

    for premises in df_rules['Premises']:
        seen_attrs = set()
        for cond in premises:
            attr = cond['attr']
            seen_attrs.add(attr)
        for attr in seen_attrs:
            attr_counter[attr] += 1  # Count only once per rule
    return dict(attr_counter)

def build_attr_usage_df(df_rules):
    rows = []
    for explainer, group in df_rules.groupby('Explainer'):
        attr_counts = count_unique_attributes_from_rules(group)
        for attr, count in attr_counts.items():
            rows.append({"Explainer": explainer, "Feature": attr, "Count": count})
    return pd.DataFrame(rows)

def plot_feature_usage_heatmap(df: pd.DataFrame,
                               feature_col: str,
                               explainer_col: str,
                               count_col: str,
                               all_features: list,
                               vmax: float = None,
                               figsize=(10,8),
                               cmap='YlOrRd'):

    if df.empty:
        print("Empty dataframe: nothing to plot.")
        return
    # pivot table with counts: rows=features, columns=explainers
    heatmap_data = df.pivot_table(index=feature_col, columns=explainer_col, values=count_col, fill_value=0)
    heatmap_data = heatmap_data.reindex(all_features, fill_value=0)

    explainers_with_features = heatmap_data.columns.tolist()
    top_feature_names = heatmap_data.index.tolist()
    actual_max_count = heatmap_data.values.max()
    theoretical_max = vmax if vmax is not None else actual_max_count

    if heatmap_data is not None and theoretical_max > 0:
        plt.figure(figsize=figsize)
        plt.imshow(heatmap_data, cmap=cmap, vmax=theoretical_max, aspect='auto')
        plt.colorbar(label='Rule Count')
        plt.xticks(ticks=range(len(explainers_with_features)), labels=explainers_with_features, rotation=45, ha='right')
        plt.yticks(ticks=range(len(top_feature_names)), labels=top_feature_names)
        plt.title(f'Feature Usage Across Explainers\n(Scale: 0 to {theoretical_max}, Actual max: {actual_max_count})')
        plt.tight_layout()
        plt.show()
    else:
        print("No data to display or vmax <= 0.")

def plot_rules_comparison(all_rules, filtered_rules, instance_name):
    plt.figure(figsize=(8,6))

    # palette for explainers across both datasets
    explainers = sorted(all_rules['Explainer'].unique())
    palette = sns.color_palette('tab10', n_colors=len(explainers))
    explainer_palette = dict(zip(explainers, palette))

    # all rules - circles
    sns.scatterplot(
        data=all_rules,
        x='Cov',
        y='Pre',
        hue='Explainer',
        palette=explainer_palette,
        s=60,
        alpha=0.5,
        marker='o'
    )

    # filtered rules - X markers with edgecolors
    sns.scatterplot(
        data=filtered_rules,
        x='Cov',
        y='Pre',
        hue='Explainer',
        palette=explainer_palette,
        s=120,
        marker='X',
        edgecolor='black'
    )

    # lines connecting filtered rules sorted by Coverage
    filtered_sorted = filtered_rules.sort_values(by='Cov')
    plt.plot(filtered_sorted['Cov'], filtered_sorted['Pre'], color='gray', linestyle='--', linewidth=1)

    plt.title(f"Rules for Instance {instance_name} (Coverage ↑, Precision ↑)")
    plt.xlabel("Coverage")
    plt.ylabel("Precision")

    handles, labels = plt.gca().get_legend_handles_labels()

    by_label = dict(zip(labels, handles))

    plt.legend(by_label.values(), by_label.keys(), title='Explainer', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.grid(True)
    plt.tight_layout()
    plt.show()

def summarize_explainer_metrics_with_global_average(explainer_dict):
    summary = {}
    all_rows = []

    for explainer, metrics_list in explainer_dict.items():
        df = pd.DataFrame(metrics_list)
        summary[explainer] = df.mean()
        all_rows.append(df)

    df_summary = pd.DataFrame(summary).T

    global_df = pd.concat(all_rows, ignore_index=True)
    df_summary.loc["Global_Average"] = global_df.mean()

    return df_summary

In [3]:
from IPython.display import display, Markdown
pd.set_option('display.max_colwidth', None)

def analyze_results(rules_log, entries_log, min_cov = 0.1, min_cov_class = 0.1, min_pre = 0.1):
    path = "./experiments_log/"
    rules_grouped = load_and_group_rules(path+rules_log)
    df_instances = load_entries_to_df(path+entries_log)
    df_rules = grouped_rules_to_df(rules_grouped)
    instance_names = df_rules['Instance_Name'].unique()
    filtered_counts = defaultdict(list)
    non_dom1_by_explainer = defaultdict(list)
    non_dom2_by_explainer = defaultdict(list)
    filtered_counts_by_explainer = {
        'Correct Prediction': defaultdict(list),
        'Threshold Filter': defaultdict(list),
        'Non-dominated 1': defaultdict(list),
        'Non-dominated 2': defaultdict(list),
    }
    agg_all_dom1 = []
    agg_all_dom2 = []

    for i, instance_name in enumerate(instance_names, start=1):
        max_rules = df_rules[df_rules['Instance_Name'] == instance_name].groupby('Explainer').size().max()
        orig = df_instances.loc[df_instances['Instance_Name'] == instance_name, "Original_Outcome"].iloc[0]
        pred = df_instances.loc[df_instances['Instance_Name'] == instance_name, "Predicted_Outcome"].iloc[0]
        display(Markdown(f"## Instance {instance_name} (Original: {orig} , Predicted: {pred})"))
        instance = df_instances[df_instances['Instance_Name'] == instance_name]
        exclude_cols = ['Instance_Name', 'Original_Outcome', 'Predicted_Outcome']
        attributes = [col for col in instance.columns if col not in exclude_cols]
        display(instance.drop(columns=['Instance_Name', 'Original_Outcome', 'Predicted_Outcome']).T.rename(columns={df_instances[df_instances['Instance_Name'] == instance_name].index[0]: 'Value'}))

        display(Markdown(f"### Rules for Instance {instance_name}"))
        display(df_rules[df_rules['Instance_Name'] == instance_name].drop(columns=["Premises"]).reset_index(drop=True))

        display(Markdown(f"### Rules for Instance {instance_name}, Correct Prediction"))
        original_outcome = df_instances.loc[df_instances['Instance_Name'] == instance_name, 'Original_Outcome'].values[0]
        correct_pred_rules = df_rules[
        (df_rules['Instance_Name'] == instance_name) &
        (df_rules['Rule'].str.contains(f'class = {original_outcome}', na=False))
        ]
        display(correct_pred_rules.drop(columns=["Premises"]).reset_index(drop=True))

        display(Markdown(f"### Rules for Instance {instance_name}, Min_treshold (Cov {min_cov}, Cov_class {min_cov_class}, Pre {min_pre})"))
        tresholded_rules = correct_pred_rules[
        (correct_pred_rules['Cov'] >= min_cov) &
        (correct_pred_rules['Cov_class'] >= min_cov_class) &
        (correct_pred_rules['Pre'] >= min_pre)
        ]
        display(tresholded_rules.drop(columns=["Premises"]).reset_index(drop=True))


        display(Markdown(f"### Rules for Instance {instance_name}, Non-dominated (Cov↑, Pre↑)"))
        non_dominated_rules1 = filter_non_dominated(tresholded_rules)
        display(non_dominated_rules1.drop(columns=["Premises"]).reset_index(drop=True))
        plot_non_dominated_rules(non_dominated_rules1, instance_name)
        plot_rules_comparison(all_rules=df_rules[df_rules['Instance_Name'] == instance_name],
                          filtered_rules=non_dominated_rules1,
                          instance_name=instance_name)
        agg_df = build_attr_usage_df(non_dominated_rules1)
        agg_all_dom1.append(agg_df)
        plot_feature_usage_heatmap(agg_df, feature_col="Feature", explainer_col="Explainer", count_col="Count", all_features=attributes, vmax = max_rules)

        display(Markdown(f"### Rules for Instance {instance_name}, Non-dominated (Cov_class↑, Pre↑, Len↓)"))
        non_dominated_rules2 = filter_non_dominated_3d(tresholded_rules)
        display(non_dominated_rules2.drop(columns=["Premises"]).reset_index(drop=True))
        agg_df = build_attr_usage_df(non_dominated_rules2)
        agg_all_dom2.append(agg_df)
        plot_feature_usage_heatmap(agg_df, feature_col="Feature", explainer_col="Explainer", count_col="Count", all_features=attributes, vmax = max_rules)

        all_rules_count = len(df_rules[df_rules['Instance_Name'] == instance_name])
        correct_pred_count = len(correct_pred_rules)
        tresholded_count = len(tresholded_rules)
        non_dom1_count = len(non_dominated_rules1)
        non_dom2_count = len(non_dominated_rules2)

        # filtered counts
        filtered_counts['Correct Prediction'].append(all_rules_count - correct_pred_count)
        filtered_counts['Threshold Filter'].append(correct_pred_count - tresholded_count)
        filtered_counts['Non-dominated 1'].append(tresholded_count - non_dom1_count)
        filtered_counts['Non-dominated 2'].append(tresholded_count - non_dom2_count)

        original_rules_instance = df_rules[df_rules['Instance_Name'] == instance_name]
        orig_counts = original_rules_instance['Explainer'].value_counts()
        correct_pred_counts = correct_pred_rules['Explainer'].value_counts()
        filtered_per_explainer_cp = (orig_counts - correct_pred_counts).fillna(orig_counts).astype(int)
        for explainer, count in filtered_per_explainer_cp.items():
            filtered_counts_by_explainer['Correct Prediction'][explainer].append(count)
        tresholded_counts = tresholded_rules['Explainer'].value_counts()
        filtered_per_explainer_thresh = (correct_pred_counts - tresholded_counts).fillna(correct_pred_counts).astype(int)
        for explainer, count in filtered_per_explainer_thresh.items():
            filtered_counts_by_explainer['Threshold Filter'][explainer].append(count)
        non_dom1_counts = non_dominated_rules1['Explainer'].value_counts()
        filtered_per_explainer_non_dom1 = (tresholded_counts - non_dom1_counts).fillna(tresholded_counts).astype(int)
        for explainer, count in filtered_per_explainer_non_dom1.items():
            filtered_counts_by_explainer['Non-dominated 1'][explainer].append(count)
        non_dominated_rules2 = filter_non_dominated_3d(tresholded_rules)
        non_dom2_counts = non_dominated_rules2['Explainer'].value_counts()
        filtered_per_explainer_non_dom2 = (tresholded_counts - non_dom2_counts).fillna(tresholded_counts).astype(int)
        for explainer, count in filtered_per_explainer_non_dom2.items():
            filtered_counts_by_explainer['Non-dominated 2'][explainer].append(count)


        if not non_dominated_rules1.empty:
            for explainer, group in non_dominated_rules1.groupby("Explainer"):
                non_dom1_by_explainer[explainer].append(
                    group[['Pre', 'Cov', 'Cov_class', 'Len', 'Reject', 'Elapsed_time']].mean()
                )

        if not non_dominated_rules2.empty:
            for explainer, group in non_dominated_rules2.groupby("Explainer"):
                non_dom2_by_explainer[explainer].append(
                    group[['Pre', 'Cov', 'Cov_class', 'Len', 'Reject', 'Elapsed_time']].mean()
                )


    # average filtered rules per stage
    display(Markdown("### Average Number of Filtered Rules at Each Step"))
    filtration_summary = pd.DataFrame({stage: pd.Series(values).mean() for stage, values in filtered_counts.items()}, index=["Avg Filtered"]).T
    # display(filtration_summary)
    step_avg_dfs = {}
    for step, explainer_dict in filtered_counts_by_explainer.items():
        df = pd.DataFrame({expl: pd.Series(counts) for expl, counts in explainer_dict.items()})
        step_avg_dfs[step] = df.mean().rename(step)  # Series with index=explainer, values=avg filtered

    combined_df = pd.concat(step_avg_dfs.values(), axis=1).fillna(0)
    combined_df.index.name = 'Explainer'
    combined_df.columns = [f"Avg Filtered ({col})" for col in combined_df.columns]

    mapping = {
        'Correct Prediction': 'Avg Filtered (Correct Prediction)',
        'Threshold Filter': 'Avg Filtered (Threshold Filter)',
        'Non-dominated 1': 'Avg Filtered (Non-dominated 1)',
        'Non-dominated 2': 'Avg Filtered (Non-dominated 2)'
    }
    new_row = {}
    for step, col_name in mapping.items():
        new_row[col_name] = filtration_summary.loc[step, 'Avg Filtered']

    combined_df.loc['Overall Average'] = pd.Series(new_row)
    display(combined_df)

    # average metrics for non-dominated rules
    display(Markdown("### Average Metrics for Non-Dominated Rules (Cov↑, Pre↑)"))
    explainer_summary1 = summarize_explainer_metrics_with_global_average(non_dom1_by_explainer)
    display(explainer_summary1)

    display(Markdown("### Average Metrics for Non-Dominated Rules (Cov_class↑, Pre↑, Len↓)"))
    explainer_summary2 = summarize_explainer_metrics_with_global_average(non_dom2_by_explainer)
    display(explainer_summary2)

    agg_all_dom1_df = pd.concat(agg_all_dom1).groupby(['Feature', 'Explainer']).sum().reset_index()
    agg_all_dom2_df = pd.concat(agg_all_dom2).groupby(['Feature', 'Explainer']).sum().reset_index()

    all_features = [col for col in df_instances.columns if col not in ['Instance_Name', 'Original_Outcome', 'Predicted_Outcome']]

    display(Markdown("## Overall Heatmap – Non-dominated Rules (Cov↑, Pre↑)"))
    plot_feature_usage_heatmap(agg_all_dom1_df, feature_col="Feature", explainer_col="Explainer", count_col="Count", all_features=all_features, vmax = max_rules*len(instance_names))

    display(Markdown("## Overall Heatmap – Non-dominated Rules (Cov_class↑, Pre↑, Len↓)"))
    plot_feature_usage_heatmap(agg_all_dom2_df, feature_col="Feature", explainer_col="Explainer", count_col="Count", all_features=all_features, vmax = max_rules*len(instance_names))

In [1]:
rules_log = "adult_random_forest_rules_36-20_06-06-2025.txt"
entries_log = "adult_random_forest_entries_36-20_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined

In [2]:
rules_log = "adult_simpleNN_rules_19-23_06-06-2025.txt"
entries_log = "adult_simpleNN_entries_19-23_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined

In [3]:
rules_log = "german_random_forest_rules_24-23_06-06-2025.txt"
entries_log = "german_random_forest_entries_24-23_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined

In [4]:
rules_log = "german_simpleNN_rules_53-23_06-06-2025.txt"
entries_log = "german_simpleNN_entries_53-23_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined

In [5]:
rules_log = "fico_heloc_random_forest_rules_12-00_06-06-2025.txt"
entries_log = "fico_heloc_random_forest_entries_12-00_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined

In [6]:
rules_log = "fico_heloc_simpleNN_rules_25-00_06-06-2025.txt"
entries_log = "fico_heloc_simpleNN_entries_25-00_06-06-2025.txt"
analyze_results(rules_log, entries_log, min_cov = 0.01, min_cov_class = 0.01, min_pre = 0.01)

NameError: name 'analyze_results' is not defined